In [ ]:
"""
Tidrapportering - Python-konvertering från SAS
Hanterar tidrapporter från TSV-fil och skapar rapporter för Antura och flextid
"""

import polars as pl
import duckdb
from datetime import datetime, timedelta
from pathlib import Path
import openpyxl
import pyarrow
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
import sys

# ============================================================================
# KONFIGURATION
# ============================================================================

INPUT_FILE = r'C:\Users\freha296\Documents\Administrativt\All_my_time.tsv'
OUTPUT_FILE = 'Tidrapport.xlsx'

# ============================================================================
# FUNKTIONER
# ============================================================================

def load_time_data(filepath: str) -> pl.DataFrame:
    """
    Läser in tidrapportdata från TSV-fil
    Motsvarar DATA WORK.All_my_time i SAS
    """
    # Läs filen rad för rad för att hantera kommatecken i texten korrekt
    import csv
    
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        # TSV-filen använder tab som separator, men SAS-koden visar DLM='2c'x vilket är komma
        # Låt oss testa med komma först, sedan tab om det inte fungerar
        f.seek(0)
        first_line = f.readline()
        
        # Räkna antal tabs vs komman i första raden
        if '\t' in first_line:
            delimiter = '\t'
        else:
            delimiter = ','
        
        f.seek(0)
        reader = csv.reader(f, delimiter=delimiter, quotechar='"')
        headers = next(reader)  # Läs headers
        
        for row in reader:
            if len(row) >= 4:  # Minst Work Item, Start, End, Duration
                rows.append(row)
    
    # Skapa DataFrame
    df = pl.DataFrame({
        'Work Item': [row[0] if len(row) > 0 else '' for row in rows],
        'Start': [row[1] if len(row) > 1 else '' for row in rows],
        'End': [row[2] if len(row) > 2 else '' for row in rows],
        'Duration': [row[3] if len(row) > 3 else '0:00:00' for row in rows],
        'Notes': [row[4] if len(row) > 4 else '' for row in rows]
    })
    
    # Konvertera datum och tid
    df = df.with_columns([
        pl.col('Start').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S', strict=False).alias('Start'),
        pl.col('End').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S', strict=False).alias('End'),
        # Duration är i format HH:MM:SS, konvertera till sekunder
        pl.col('Duration').str.split(':').map_elements(
            lambda x: int(x[0]) * 3600 + int(x[1]) * 60 + int(x[2]) if len(x) == 3 else 0,
            return_dtype=pl.Int64
        ).alias('Duration')
    ])
    
    return df

print("Box 1: Klar")


Box 1: Klar


In [49]:
import sys
print(sys.executable)

c:\Program Files\Python313\python.exe


In [30]:
def add_derived_columns(df: pl.DataFrame) -> pl.DataFrame:
    """
    Lägger till härledda kolumner
    Motsvarar WORK.DERIVED_COLUMNS i SAS
    """
    df = df.with_columns([
        pl.col('Start').dt.date().alias('startdatum'),
        pl.col('Start').dt.month().alias('manad')
    ])
    
    return df


In [31]:
def validate_single_month(df: pl.DataFrame) -> None:
    """
    Kontrollerar att filen endast innehåller en månad
    Motsvarar varna_om_antal_manader makrot
    """
    stats = df.select([
        pl.col('startdatum').min().alias('min_datum'),
        pl.col('startdatum').max().alias('max_datum')
    ]).row(0)
    
    min_datum, max_datum = stats
    
    # Kontrollera om de är i samma månad
    if min_datum.month != max_datum.month or min_datum.year != max_datum.year:
        print(f"ERROR: Data innehåller flera månader ({min_datum} till {max_datum})")
        print("ERROR: Sessionen avbryts")
        sys.exit(1)
    
    print(f"Data validerad: {min_datum} till {max_datum}")


In [32]:
def create_all_days(df: pl.DataFrame) -> pl.DataFrame:
    """
    Skapar upp alla dagar för att fylla ut dataset (för lediga dagar)
    Motsvarar data alla_dagar i SAS
    """
    min_datum = df.select(pl.col('startdatum').min()).item()
    max_datum = df.select(pl.col('startdatum').max()).item()
    
    # Första och sista dagen i månaden
    from dateutil.relativedelta import relativedelta
    forsta = min_datum.replace(day=1)
    sista = (max_datum.replace(day=1) + relativedelta(months=1)) - timedelta(days=1)
    
    # Skapa alla datum
    date_range = pl.date_range(forsta, sista, interval='1d', eager=True)
    
    all_days = pl.DataFrame({
        'startdatum': date_range,
        'Duration': [0] * len(date_range)
    })
    
    return all_days



In [35]:
def create_main_query(df: pl.DataFrame, all_days: pl.DataFrame) -> pl.DataFrame:
    """
    Skapar huvuddataset med alla beräknade kolumner
    Motsvarar WORK.QUERY_FOR_THISMONTH_DERIVED_COLU i SAS
    """
    # Full outer join mellan tiddata och alla dagar
    merged = df.join(all_days, on='startdatum', how='full', suffix='_all')
    
    merged = merged.with_columns([
        # Duration - använd från tiddata om den finns, annars från all_days
        pl.when(pl.col('Duration').is_not_null())
          .then(pl.col('Duration'))
          .otherwise(pl.col('Duration_all'))
          .alias('Duration'),
        
        # DurationTimmar - konvertera sekunder till timmar
        (pl.coalesce(['Duration', 'Duration_all']) / 3600.0).alias('DurationTimmar'),
        
        # startdatum - använd från tiddata om den finns
        pl.coalesce(['startdatum', 'startdatum_all']).alias('startdatum')
    ])
    
    # Lägg till vecka, veckodag, år etc
    merged = merged.with_columns([
        pl.col('startdatum').dt.week().alias('Vecka'),
        pl.col('startdatum').dt.strftime('%a').alias('veckodag'),
        pl.col('startdatum').dt.month().alias('manad'),
        pl.col('startdatum').dt.year().alias('ar')
    ])
    
    # Välj relevanta kolumner
    cols = ['Work Item', 'Start', 'End', 'Duration', 'DurationTimmar', 
            'startdatum', 'Vecka', 'veckodag', 'manad', 'ar', 'Notes']
    
    return merged.select([c for c in cols if c in merged.columns])



In [37]:
def add_time_type(df: pl.DataFrame) -> pl.DataFrame:
    """
    Lägger till kategori för typ av tid
    Motsvarar WORK.typ_av_tid i SAS
    """
    twelve_months_ago = datetime.now().date() - timedelta(days=365)
    
    df = df.filter(pl.col('startdatum') > twelve_months_ago)
    
    df = df.with_columns([
        pl.when(~pl.col('Work Item').str.starts_with('Från'))
          .then(pl.lit('Arbetstid'))
          .otherwise(pl.col('Work Item'))
          .alias('Typ_av_tid')
    ])
    
    return df



In [38]:
def create_antura_report(df: pl.DataFrame, output_file: str):
    """
    Skapar Antura-rapport i Excel
    Motsvarar PROC TABULATE för Antura i SAS
    """
    # Filtrera till förra månaden
    from dateutil.relativedelta import relativedelta
    today = datetime.now().date()
    first_day_last_month = (today.replace(day=1) - relativedelta(months=1))
    
    df_filtered = df.filter(pl.col('startdatum') >= first_day_last_month)
    
    # Skapa Excel-arbetsbok
    wb = openpyxl.Workbook()
    
    # Ta bort standard-sheet
    if 'Sheet' in wb.sheetnames:
        wb.remove(wb['Sheet'])
    
    # Hämta unika veckor
    weeks = df_filtered.select(['ar', 'Vecka']).unique().sort(['ar', 'Vecka'])
    
    for row in weeks.iter_rows():
        ar, vecka = row
        
        # Skapa nytt sheet för varje vecka
        ws = wb.create_sheet(title=f'Vecka {vecka}')
        
        # Filtrera data för denna vecka
        week_data = df_filtered.filter(
            (pl.col('ar') == ar) & (pl.col('Vecka') == vecka)
        )
        
        # Skapa pivot-tabell med DuckDB
        conn = duckdb.connect()
        conn.register('week_data', week_data.to_pandas())
        
        pivot_query = """
        SELECT 
            "Work Item",
            SUM(CASE WHEN veckodag = 'Mon' THEN DurationTimmar ELSE 0 END) as Mån,
            SUM(CASE WHEN veckodag = 'Tue' THEN DurationTimmar ELSE 0 END) as Tis,
            SUM(CASE WHEN veckodag = 'Wed' THEN DurationTimmar ELSE 0 END) as Ons,
            SUM(CASE WHEN veckodag = 'Thu' THEN DurationTimmar ELSE 0 END) as Tor,
            SUM(CASE WHEN veckodag = 'Fri' THEN DurationTimmar ELSE 0 END) as Fre,
            SUM(CASE WHEN veckodag = 'Sat' THEN DurationTimmar ELSE 0 END) as Lör,
            SUM(CASE WHEN veckodag = 'Sun' THEN DurationTimmar ELSE 0 END) as Sön,
            SUM(DurationTimmar) as Veckosumma
        FROM week_data
        WHERE "Work Item" IS NOT NULL
        GROUP BY "Work Item"
        ORDER BY "Work Item"
        """
        
        pivot_df = conn.execute(pivot_query).fetchdf()
        
        # Lägg till dagsummor
        day_totals = ['Dagsumma'] + [
            pivot_df[col].sum() for col in pivot_df.columns[1:]
        ]
        
        # Skriv till Excel
        ws.append(['Vecka ' + str(vecka)])
        ws.append([])
        
        # Headers
        ws.append(['Work Item'] + list(pivot_df.columns[1:]))
        
        # Data
        for row in pivot_df.itertuples(index=False):
            ws.append(list(row))
        
        # Dagsummor
        ws.append(day_totals)
        
        # Formatera
        for row in ws.iter_rows(min_row=3, max_row=3):
            for cell in row:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='CCCCCC', fill_type='solid')
        
        # Formatera siffror
        for row in ws.iter_rows(min_row=4, min_col=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
        
        # Anteckningar
        notes_data = week_data.filter(pl.col('Notes').is_not_null() & (pl.col('Notes') != ''))
        
        if len(notes_data) > 0:
            ws.append([])
            ws.append(['Anteckningar'])
            ws.append(['Datum', 'Anteckning', 'Timmar'])
            
            for row in notes_data.select(['startdatum', 'Notes', 'DurationTimmar']).iter_rows():
                ws.append(list(row))
        
        conn.close()
    
    # Spara
    wb.save(output_file)
    print(f"Antura-rapport sparad: {output_file}")



In [39]:
def create_flex_report(df: pl.DataFrame, output_file: str):
    """
    Skapar flextidsrapport
    Motsvarar PROC TABULATE för flex i SAS
    """
    # Använd DuckDB för aggregering
    conn = duckdb.connect()
    conn.register('df', df.to_pandas())
    
    query = """
    SELECT 
        manad,
        startdatum,
        veckodag,
        Typ_av_tid,
        SUM(Duration) as Total_Duration
    FROM df
    GROUP BY manad, startdatum, veckodag, Typ_av_tid
    ORDER BY manad, startdatum, Typ_av_tid
    """
    
    result = pl.from_pandas(conn.execute(query).fetchdf())
    conn.close()
    
    # Konvertera Duration tillbaka till HH:MM format
    result = result.with_columns([
        (pl.col('Total_Duration') // 3600).cast(pl.Int32).alias('hours'),
        ((pl.col('Total_Duration') % 3600) // 60).cast(pl.Int32).alias('minutes')
    ])
    
    result = result.with_columns([
        (pl.col('hours').cast(pl.Utf8) + ':' + 
         pl.col('minutes').cast(pl.Utf8).str.zfill(2)).alias('Duration_Formatted')
    ])
    
    print("\n=== FLEXTIDSRAPPORT ===")
    print(result.select(['manad', 'startdatum', 'veckodag', 'Typ_av_tid', 'Duration_Formatted']))
    
    return result



In [48]:

# ============================================================================
# HUVUDPROGRAM
# ============================================================================

def main():
    print("=== Tidrapportering - Python version ===\n")
    
    # 1. Läs in data
    print("Läser in data...")
    df = load_time_data(INPUT_FILE)
    print(f"Läste {len(df)} rader")

    # 2. Lägg till härledda kolumner
    print("\nLägger till härledda kolumner...")
    df = add_derived_columns(df)

    # print(df.head())

    
    # 3. Validera att endast en månad finns
    # print("\nValiderar data...")
    # validate_single_month(df)

    # 4. Skapa alla dagar
    print("\nSkapar alla dagar i månaden...")
    all_days = create_all_days(df)

    # print(all_days.head())

    # 5. Skapa huvuddataset
    print("\nSkapar huvuddataset...")
    main_df = create_main_query(df, all_days)

    # print(main_df.head())

    # 6. Lägg till typ av tid
    print("\nLägger till tidskategorier...")
    type_df = add_time_type(main_df)

    # print(type_df.head())


    # 7. Skapa Antura-rapport
    print("\nSkapar Antura-rapport...")
    create_antura_report(main_df, OUTPUT_FILE)


def senare():
    # 8. Skapa flextidsrapport
    print("\nSkapar flextidsrapport...")
    create_flex_report(type_df, 'Flextid_rapport.xlsx')
    
    print("\n=== Klart! ===")


if __name__ == "__main__":
    main()


=== Tidrapportering - Python version ===

Läser in data...
Läste 1770 rader

Lägger till härledda kolumner...

Skapar alla dagar i månaden...

Skapar huvuddataset...

Lägger till tidskategorier...

Skapar Antura-rapport...


ModuleNotFoundError: pa.Table requires 'pyarrow' module to be installed